Connected to criticality_env (Python 3.13.0)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import statsmodels.api as sm
import numpy as np


# all following functions require as input a joint dataframe of results created with the function: tools.mr_estimator_for_numpy()
# pandas dataframe with columns:
    # categorical: area, activity, task, treatment
    # numerical: tau, timechunk


def read_mr_estimator_results(result_directory, output_directory = None):
    """made output_directory optional. But highly recommended since integration of all data may take some time

    Args:
        result_directory (str): _description_
        output_directory (_type_, optional): _description_. Defaults to None.

    Returns:
        _type_: _description_
    """
    files = [f for f in os.listdir(result_directory) if f.endswith(".parquet")]

    all_data = []
    for filename in files:
        filename_split = filename.split('_') # with filename pattern being:  "animal name _ experimental day _ task _ tetrode nr. _ brain area _ pre or post treatment _ timechunk .parquet"
        
        df = pd.read_parquet(f'{result_directory}/{filename}')
        df['tau'] = df['tau'][0]
        df['animal'] = filename_split[0]
        df['day'] = filename_split [1]
        df ['task'] = filename_split[2]
        df['area'] = filename_split[3]
        df['treatment'] = filename_split[4][:-4]
        df['timechunk'] = int(filename_split[5].split('.')[0])
        all_data.append(df)

    # Combine all data
    df = pd.concat(all_data, ignore_index=True)

    # Convert categorical variables
    df['area'] = df['area'].astype('category')
    df['treatment'] = df['treatment'].astype('category')
    df['animal'] = df['animal'].astype('category')
    df['task'] = df['task'].astype('category')

    if output_directory:
        df.to_parquet(f'{output_directory}big_df.parquet')
    
    return df


def tau_for_area_prepost_violin(df, output_directory: str = None, add_filename: str = None):
    
    # Set plot style
    sns.set(style="whitegrid")

    # Filter data to include only tau values <= 3000
    filtered_df = df[df['tau'] <= 3000]

    # Create the violin plot with reduced smoothing
    plt.figure(figsize=(10, 6))
    sns.violinplot(
        data=filtered_df,
        x='area',
        y='tau',
        hue='treatment',
        bw=0.2,  # Reduce smoothing
        cut=0,   # Restrict violins to data range
        split=True,
        palette='muted'
    )

    # Add labels and title
    plt.title('Tau Distributions by Brain Area and Treatment (Filtered at 3000)', fontsize=16)
    plt.xlabel('Brain Area', fontsize=14)
    plt.ylabel('Tau', fontsize=14)
    plt.legend(title='Treatment')
    plt.tight_layout()

    if output_directory:
        plt.savefig(f"{output_directory}violin_prepost_areas_{add_filename}.png")
        
    # Show plot
    plt.show()


def tau_over_own_activity_scatter(df, output_directory = None):
    unique_areas = df['area'].unique()

    # Create scatter plots for each area with regression lines
    for area in unique_areas:
        plt.figure(figsize=(8, 6))
        
        # Filter data for the specific brain area
        area_data = df[df['area'] == area]
        
        # Fit a regression line
        x = area_data['activity']
        y = area_data['tau']
        if len(x) > 1:  # Ensure enough data points for regression
            x_with_const = sm.add_constant(x)
            model = sm.OLS(y, x_with_const).fit()
            slope = model.params['activity']
            intercept = model.params['const']
            label = f"y = {intercept:.2f} + {slope:.2f}x\nn = {len(x)}"
        else:
            slope, intercept = np.nan, np.nan
            label = f"Insufficient data (n={len(x)})"
        
        # Scatter plot
        sns.scatterplot(data=area_data, x='activity', y='tau', alpha=0.7, label="Data points")
        
        # Add regression line if valid
        if not np.isnan(slope):
            plt.plot(x, intercept + slope * x, color='red', label=f"Regression: {label}")
        
        # Add titles and labels
        plt.title(f"Scatter Plot with Regression for {area}", fontsize=16)
        plt.xlabel("Activity", fontsize=14)
        plt.ylabel("Tau", fontsize=14)
        plt.legend(fontsize=12)
        
        # Save the plot
        plt.tight_layout()
        plt.savefig(f"{output_directory}scatter_with_regression_{area}.png")
        plt.close()

    # Create combined scatter plot with regression lines for all areas
    plt.figure(figsize=(10, 8))

    # Scatter plot
    sns.scatterplot(data=df, x='activity', y='tau', hue='area', alpha=0.7, palette='muted', legend="full")

    # Titles and labels
    plt.title("Combined Scatter Plot for All Brain Areas", fontsize=16)
    plt.xlabel("Activity", fontsize=14)
    plt.ylabel("Tau", fontsize=14)

    # Add annotations for each area
    for area in unique_areas:
        area_data = df[df['area'] == area]
        x = area_data['activity']
        y = area_data['tau']
        if len(x) > 1:
            x_with_const = sm.add_constant(x)
            model = sm.OLS(y, x_with_const).fit()
            slope = model.params['activity']
            intercept = model.params['const']
            plt.plot(x, intercept + slope * x, label=f"{area}: y = {intercept:.2f} + {slope:.2f}x\nn = {len(x)}")
        else:
            plt.annotate(f"{area} (n={len(x)})", xy=(x.mean(), y.mean()), fontsize=10)

    # Adjust legend
    plt.legend(title="Brain Area", bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)

    # Save the combined plot
    plt.tight_layout()
    if output_directory:
        plt.savefig(f"{output_directory}scatter_combined_with_regression.png")
    plt.show()
    return
    
    
def tau_over_ca2_activity_scatter(df, store_directory = None):
    
    # Filter CA2 data and rename 'activity' for clarity
    ca2_data = df[df['area'] == "CA2"].rename(columns={'activity': 'activity_CA2'})

    # Merge CA2 activity with the original dataframe
    merged_df = pd.merge(
        df,
        ca2_data[['animal', 'day', 'task', 'timechunk', 'activity_CA2']],
        on=['animal', 'day', 'task', 'timechunk'],
        how='inner'
        )

    # Exclude rows where area is CA2 (since we're comparing other areas to CA2)
    comparison_df = merged_df[merged_df['area'] != "CA2"]
    
    # Get unique areas to create scatter plots
    unique_areas = comparison_df['area'].unique()

    # Create scatter plots for each area
    for area in unique_areas:
        plt.figure(figsize=(8, 6))
        
        # Filter data for the specific brain area
        area_data = comparison_df[comparison_df['area'] == area]
        
        # Fit a regression line
        x = area_data['activity_CA2']
        y = area_data['tau']
        if len(x) > 1:  # Ensure enough data points for regression
            x_with_const = sm.add_constant(x)
            model = sm.OLS(y, x_with_const).fit()
            slope = model.params['activity_CA2']
            intercept = model.params['const']
            label = f"y = {intercept:.2f} + {slope:.2f}x\nn = {len(x)}"
        else:
            slope, intercept = np.nan, np.nan
            label = f"Insufficient data (n={len(x)})"
        
        # Scatter plot
        sns.scatterplot(
            data=area_data, 
            x='activity_CA2', 
            y='tau', 
            alpha=0.7, 
            #label= 'Data points',
            hue = 'treatment',
            palette={'pre': 'blue', 'post': 'orange'})
        
        # Add regression line if valid
        if not np.isnan(slope):
            plt.plot(x, intercept + slope * x, color='red', label=f"Regression: {label}")
        
        # Add titles and labels
        plt.title(f"Scatter Plot Comparing {area} with CA2", fontsize=16)
        plt.xlabel("Activity in CA2", fontsize=14)
        plt.ylabel(f"Tau in {area}", fontsize=14)
        plt.legend(fontsize=12)
        plt.tight_layout()
        
        if store_directory:
            plt.savefig(f"{store_directory}scatter_comparison_CA2_{area}.png")
    
    return
    




df = pd.read_parquet("/cns/share/Vincent/McHugh_lab/CA2Cre_DREADD/models_and_figures/big_df.parquet")
output_dir = "/cns/share/Vincent/McHugh_lab/CA2Cre_DREADD/models_and_figures/"

In [ ]:
unique_tasks = df['task'].unique()
print(unique_tasks)

['sleep3', 'cno1', 'cno?', 'lt12', 'sleep1', ..., 'boxA2', 'hab2', 'boxB1', 'sleep5', 'boxC1']
Length: 20
Categories (20, object): ['boxA1', 'boxA2', 'boxB1', 'boxC1', ..., 'sleep2', 'sleep3', 'sleep4', 'sleep5']


In [ ]:
df['task'] = df['task'].str.slice(0, -2)
unique_tasks = df['task'].unique()
print(unique_tasks)

['slee' 'cn' 'lt' 'sle' 'box' 'ha']


In [ ]:
df = df[df['task'] != 'ha']

In [ ]:
df['task'] = df['task'].replace({'slee': 'sleep', 'cn': 'cno', 'sle': 'sleep'})

In [ ]:
df['task'] = df['task'].cat.remove_unused_categories()

AttributeError: Can only use .cat accessor with a 'category' dtype

In [ ]:
df['task'] = df['task'].astype('category')

In [ ]:
model = MixedLM.from_formula("tau ~ treatment * area * task", # predict tau based on treatment, area, and their interaction
                             groups="animal", # random effect for each animal
                             re_formula="~ area", # random slopes (area effect on tau can vary by animal/day)
                             data=df)
result = model.fit()

# 4. Summarize Resultsd
print(result.summary())

NameError: name 'MixedLM' is not defined

In [ ]:
from statsmodels.regression.mixed_linear_model import MixedLM
model = MixedLM.from_formula("tau ~ treatment * area * task", # predict tau based on treatment, area, and their interaction
                             groups="animal", # random effect for each animal
                             re_formula="~ area", # random slopes (area effect on tau can vary by animal/day)
                             data=df)
result = model.fit()

# 4. Summarize Resultsd
print(result.summary())

LinAlgError: Singular matrix

In [ ]:
from statsmodels.regression.mixed_linear_model import MixedLM
model = MixedLM.from_formula("tau ~ treatment * area * task", # predict tau based on treatment, area, and their interaction
                             groups="animal", # random effect for each animal
                             re_formula="~ area", # random slopes (area effect on tau can vary by animal/day)
                             data=df)
result = model.fit()

# 4. Summarize Resultsd
print(result.summary())

LinAlgError: Singular matrix

In [ ]:
# Drop rows with missing data in the relevant columns
df = df.dropna(subset=['tau', 'treatment', 'area', 'task', 'animal'])

# Remove categories with too few observations
min_observations = 5  # Set the minimum number of observations required

# Filter out categories in 'area' with too few observations
area_counts = df['area'].value_counts()
areas_to_keep = area_counts[area_counts >= min_observations].index
df = df[df['area'].isin(areas_to_keep)]

# Filter out categories in 'task' with too few observations
task_counts = df['task'].value_counts()
tasks_to_keep = task_counts[task_counts >= min_observations].index
df = df[df['task'].isin(tasks_to_keep)]

# Filter out categories in 'treatment' with too few observations
treatment_counts = df['treatment'].value_counts()
treatments_to_keep = treatment_counts[treatment_counts >= min_observations].index
df = df[df['treatment'].isin(treatments_to_keep)]

In [ ]:
model = MixedLM.from_formula("tau ~ treatment * area * task", # predict tau based on treatment, area, and their interaction
                             groups="animal", # random effect for each animal
                             re_formula="~ area", # random slopes (area effect on tau can vary by animal/day)
                             data=df)

In [ ]:
result = model.fit()
print(result.summary())

LinAlgError: Singular matrix

In [ ]:
print(df)

                                                   popt     ssres  \
0     [78.42746413057257, 0.03415812917882224, -1.54...  0.115052   
1     [120.31255356110354, 0.0348522019718514, -0.00...  0.121734   
2     [1009.2635184400931, 0.023504045289931, -0.009...  0.115746   
3     [182.82912223761508, 0.03460506804339023, 0.00...  0.106366   
4     [13.754500262654838, 0.1240744768629935, -9.63...  0.101305   
...                                                 ...       ...   
9239  [147.95573082523194, 0.0023719630556396495, 0....  0.115825   
9240  [159.53351486458422, 0.04247837363297372, -0.0...  0.164377   
9241  [126.34481639545525, 0.0029417280185554473, 0....  0.117369   
9242  [279.9269020216698, 0.023542176031673898, 0.01...  0.186244   
9243  [4999.99999999993, 0.31606246615999795, -0.252...  0.412824   

                                                  steps  dt dtunit quantiles  \
0     [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...   1     ms      None   
1     [1, 2